In [ ]:
# we imported required libraries 
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import accuracy_score
import numpy as np 
import pandas as pd 
import seaborn as sns

%matplotlib inline
import plotly.graph_objs as go
from plotly.subplots import make_subplots


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# We suppressed warning messages. This helps keep the output clean, especially if there are non-critical warnings.

import warnings
warnings.filterwarnings('ignore')

# Step 1: Data Collection

* Dataset Loaded: Data was read into a Pandas DataFrame.
* Initial Checks: Verified structure (df.head, df.shape) and data types (int64 for all attributes).
* No Missing Values: All rows are complete and usable.
  
* Key Observations:
Negative values in Vertical_Distance_To_Hydrology.
One-hot encoded Wilderness_Area and Soil_Type.
Constant attributes Soil_Type7 and Soil_Type15 can be removed.
Varying feature scales require normalization.

* Continuous Features Identified: Prepared a list of numerical features for preprocessing.


In [ ]:
# We loaded the dataset
df = pd.read_csv("/kaggle/input/forest-cover-type-prediction/train.csv")

# To confirm the dataset is loaded correctly and understand its basic structure, we got a quick look at the data
print(df.head())
print(df.shape)

# Datatypes of the attributes
print(df.dtypes)
# We observed data types of all attributes are int64. This information gave us an idea of what to do during the data cleaning and preprocessing stage.

print(df.describe())

print(df.isnull().values.any()) 
#printed false so no missing values

# No attribute is missing as count is 15120 for all attributes. Hence, all rows can be used
# Negative value(s) present in Vertical_Distance_To_Hydrology. Hence, some tests such as chi-sq cant be used.
# Wilderness_Area and Soil_Type are one hot encoded. Hence, they could be converted back for some analysis
# Attributes Soil_Type7 and Soil_Type15 can be removed as they are constant
# Scales are not the same for all.

# We prepared a list of features for focused analysis or preprocessing steps.
continuous_features = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 
                       'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 
                       'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 
                       'Horizontal_Distance_To_Fire_Points']



# Step 2: Data Preprocessing and Cleaning

* In this step, we focused on cleaning and preprocessing the dataset to ensure it is ready for modeling. First, we detected and removed outliers using the Z-score method, as it is efficient and straightforward for large datasets. Specifically, we visualized the distribution of the critical variable Elevation across Cover_Type categories to identify anomalies. Outliers were dropped to maintain data integrity and avoid introducing noise, as imputation was unnecessary due to the absence of missing values.

* Next, we applied the MinMaxScaler to normalize the continuous features, scaling their values to a range of 0 to 1. This ensures uniformity and prevents features with larger magnitudes from dominating the modeling process. Additionally, constant columns with no variability (Soil_Type7, Soil_Type8, Soil_Type15) were identified and dropped, as they provided no predictive value. Finally, we confirmed the changes by inspecting the updated dataset shape and a sample of the cleaned data.



In [ ]:
# Since there were no missing values, we did not perform any additional processing in this regard.

# To clean the data, firstly we detected outliers. We used the Z-score method to perform this operation.
#Here's why we chose this method among others: the Z-score method is straightforward, computationally efficient, 
#and easy to implement, especially on large datasets. 

z_scores = df[continuous_features].apply(zscore)
outliers = (z_scores.abs() > 3).any(axis=1)
print(f"Number of outliers: {outliers.sum()}")

# Elevation is a critical continuous variable likely tied to the target variable, with no strict boundaries and 
# significant variability. These characteristics make it a prime candidate for outlier detection, ensuring that
# anomalies do not adversely affect the analysis or modeling process. So we visualized the distribution of Elevation 
# values for each Cover_Type category to identify anomalies.

fig = px.box(df,x="Cover_Type",y="Elevation",color="Cover_Type",height=400,width=900)
fig.update_layout(title={'text':"Elevation Box Plot"})
fig.show()

# We removed outliers by dropping them instead of performing imputation. The reason is that imputation is used to
# fill missing or incomplete data. However, in this case, there were no missing values, and the data in the columns
# was either constant or lacked meaningful information. Calculating the mean, median, or predicting values for such
# columns is unnecessary and would only introduce noise into the dataset. Therefore, dropping these outliers is a 
# faster and more effective approach. Imputation should only be applied to meaningful features with missing or incomplete values.

# We printed the shape of the DataFrame after removing outliers to confirm changes.

df = df[~outliers]
print(df.shape)

# We did this by initializing the MinMaxScaler to scale the values to a range of 0 to 1.
# We applied the scaler to the continuous features to ensure uniformity in their scales, which benefits many machine learning models.
# Scaling ensures that all continuous features contribute equally to the model by standardizing their range. 
# Without scaling, features with larger magnitudes (e.g., Elevation) could dominate those with smaller ranges (e.g., Slope), 
# leading to biased or inefficient models.


# Initialize the scaler
scaler = MinMaxScaler()

# Applied normalization to continuous features
df[continuous_features] = scaler.fit_transform(df[continuous_features])


rem = []

# Because of the standard deviation of some columns are 0 (it means they are constant and unhelpful in prediction process.)
#we merged them to drop.

for c in df.columns:
    if df[c].std() == 0: #standard deviation is zero
        rem.append(c)

# Columns are dropped      
df.drop(rem,axis=1,inplace=True)

print(rem)
# Constant columns removed from the dataset 'Soil_Type7' 'Soil_Type8' 'Soil_Type15'

# We checked final data shape and head
print(df.head(10))
print(df.shape)

 # Step 3: Data Exploration and Analysis
We analyzed the dataset and identified distinct patterns in Cover_Type, Wild Areas, and Soil Types, highlighting dominant categories like Wilderness_Area3 and Soil_Type10. A box plot confirmed outlier removal, and the correlation map revealed key relationships, such as between Horizontal_Distance_To_Hydrology and Vertical_Distance_To_Hydrology.

Using the elbow method, we determined 4 optimal clusters. PCA visualization showed clear separations, and cluster centers revealed distinct profiles. The clusters were balanced, ensuring meaningful segmentation for further analysis.


In [ ]:
fig = px.histogram(df,x="Cover_Type",color="Cover_Type",height=400,width=800)
fig.show()

"""
The histogram shows a balanced distribution of Cover_Type categories, 
with each class having a similar count of around 2000 samples.

"""

In [ ]:
df['Wild Areas'] = (df.iloc[:,11:15] == 1).idxmax(1)
df['Soil types'] = (df.iloc[:,15:55] == 1).idxmax(1)


fig = px.pie(df,names="Wild Areas",height=300,width=800)
fig.show()

fig = px.histogram(df,x="Wild Areas",color="Cover_Type",barmode="group",
                   height=400,width=800)
fig.show()

"""
The visualizations highlight the distribution of Wild Areas and their relationship with Cover_Type.
Wilderness_Area3 is the most common (42.8%), followed by Wilderness_Area4 (31.8%) and Wilderness_Area1 (22%),
while Wilderness_Area2 is underrepresented (3.35%).

The histogram shows distinct Cover_Type patterns across Wild Areas. For example,
Wilderness_Area4 is dominated by Cover_Type 4, while Wilderness_Area3 and Wilderness_Area1 have more balanced distributions.
The low representation of Wilderness_Area2 may require special attention during modeling. 
These findings emphasize the spatial dependence of Cover_Type on Wild Areas.2. 

"""

In [ ]:
fig = px.histogram(df,x="Soil types",color="Cover_Type",height=400,width=850)
fig.show()

fig = px.pie(df,names="Soil types",height=400,width=850)
fig.update_traces(textposition='inside')
fig.show()

"""
The visualizations highlight the distribution of Soil types and their relationship with Cover_Type.
The histogram shows an uneven distribution, with Soil_Type10 being the most dominant and associated with multiple 
Cover_Type categories. Some soil types, such as Soil_Type23 and Soil_Type32, are less common and tied to specific
Cover_Type groups, indicating their unique importance in certain classifications.

The pie chart further emphasizes the prevalence of Soil_Type10, which makes up 14% of the dataset,
followed by Soil_Type29 (7.68%) and Soil_Type3 (6.83%). Many other soil types contribute less than 5% individually,
with some under 2.5%. This uneven distribution highlights the need to consider the dominant and rare soil types when
modeling to ensure proper representation and prediction accuracy
"""

In [ ]:
fig = px.box(df,x="Cover_Type",y="Elevation",color="Cover_Type",height=400,width=900)
fig.update_layout(title={'text':"Elevation Box Plot"})
fig.show()

"""
The box plot visualizes the distribution of Elevation across different Cover_Type categories, showing their distinct
ranges and variability. This plot was also used to verify whether the data was successfully cleared of outliers
during preprocessing.

"""

In [ ]:
df_corr = df[continuous_features].corr()
fig=plt.figure(figsize=(12,10))
sns.heatmap(df_corr,annot=True,linewidths=.3,cmap='YlOrBr')

"""
This correlation map showed us that are strong correlations between Horizontal_Distance_To_Hydrology and 
Vertical_Distance_To_Hydrology (0.6) and also between Horizontal_Distance_To_Roadways and 
Horizontal_Distance_To_Fire_Points. (0.48)


Negative correlations are Slope and Hillshade_Noon (-0.58), Hillshade_9am and Hillshade_3pm (-0.8).
We observed weak correlation between Aspect and Vertical_Distance_To_Hydrology. They are mostly independent.

"""

In [ ]:
inertia = []
for k in range(1, 11):  # Test 1 to 10 clusters
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df[continuous_features])
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.title("Elbow Method for Optimal Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

"""
The elbow plot indicates the optimal number of clusters is around k=4, where the inertia drop slows significantly.
Adding more clusters beyond this point offers diminishing returns in clustering quality

"""

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['Cluster'] = kmeans.fit_predict(df[continuous_features])

cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=continuous_features)
print(cluster_centers)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(df[continuous_features])

plt.scatter(pca_result[:, 0], pca_result[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.7)
plt.title("Clusters Visualized in 2D Space")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="Cluster")
plt.show()

sns.countplot(x='Cluster', data=df)
plt.title("Cluster Distribution")
plt.show()

"""
Cluster Centers:

Each cluster center provides the average values of the features for that cluster, showing their distinct profiles.
For example:
Cluster 0: Low Elevation and Horizontal_Distance_To_Roadways with moderate Hillshade_9am.
Cluster 3: High Elevation, Horizontal_Distance_To_Roadways, and Horizontal_Distance_To_Fire_Points, indicating a different spatial pattern.

2D Visualization with PCA:

PCA reduces high-dimensional data into two principal components while preserving maximum variance.
The scatter plot shows clear separations between clusters. Overlaps are minimal.
It shows that the clustering algorithm worked effectively. 

Cluster Distribution:

The bar chart shows that clusters are fairly balanced, with slight variations in size:
Cluster 0 and Cluster 2 have more samples, representing dominant data patterns.
Cluster 1 and Cluster 3 are smaller but still significant, representing less frequent patterns.

"""

# **Step 4: Predictive Modelling**

In this step, we aim to classify the target variable Cover_Type using multiple classification algorithms. We will:

Define features and target variables.
Split the data into training, validation, and test sets.
Train and evaluate Logistic Regression and Random Forest models.
Perform hyperparameter tuning for Random Forest and evaluate the optimized model.
Compare the performance of these models using appropriate metrics.

In [ ]:
print(f"df {df.shape}")

# Define target and features
X = df.drop(columns=['Cover_Type'])  # 'Cover_Type' is our target so we dropped 
y = df['Cover_Type']  # target value

# Ensure proper encoding for categorical variables
# get_dummies() function converts categorical variables into dummy/indicator variables
X = pd.get_dummies(X, drop_first=True)

# Split data into train, validation, and test sets => 70%, 10%, 20%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.66, random_state=42, stratify=y_temp)

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")

# Logistic Regression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)
log_preds = log_model.predict(X_val)
print("Logistic Regression Validation Performance:")
print(classification_report(y_val, log_preds))

# Logistic Regression: Evaluate on Test Set
log_test_preds = log_model.predict(X_test)
print("Logistic Regression Test Set Performance:")
print(classification_report(y_test, log_test_preds))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)
print("Random Forest Validation Performance:")
print(classification_report(y_val, rf_preds))

# Random Forest: Evaluate on Test Set
rf_test_preds = rf_model.predict(X_test)
print("Random Forest Test Set Performance:")
print(classification_report(y_test, rf_test_preds))


# Hyperparameter Tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Best Parameters for Random Forest: {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_}")

# Retrain Best Random Forest Model on Training + Validation Set
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

best_rf_model = RandomForestClassifier(**grid_search.best_params_, random_state=42)
best_rf_model.fit(X_train_val, y_train_val)

# Evaluate Best Random Forest Model on Test Set
best_rf_test_preds = best_rf_model.predict(X_test)
print("Best Random Forest Test Set Performance:")
print(classification_report(y_test, best_rf_test_preds))

"""

Algorithms Used:
Logistic Regression: Simple and interpretable baseline.
Random Forest: Ensemble method with strong performance.

Parameter Tuning:
Grid search is applied to Random Forest for hyperparameter optimization.

Performance Metrics:
Validation and test sets are evaluated using classification reports.
Metrics include precision, recall, F1-score, and accuracy

"""

# K-Nearest Neighbors (KNN)

K-Nearest Neighbors (KNN) is used to classify the target variable Cover_Type. 

We:
Train and evaluate the KNN model on validation and test sets.
Generate a confusion matrix for the test set.
Perform hyperparameter tuning to optimize KNN performance.

In [ ]:
# Initialize KNN classifier
knn_model = KNeighborsClassifier(n_neighbors=3)  
knn_model.fit(X_train, y_train)

# Predictions on validation set
knn_preds = knn_model.predict(X_val)

# Print classification report for KNN
print("KNN Performance on Validation Set:")
print(classification_report(y_val, knn_preds))

# Evaluate on test set
knn_test_preds = knn_model.predict(X_test)
print("KNN Test Performance:")
print(classification_report(y_test, knn_test_preds))

# Confusion Matrix for KNN
cm = confusion_matrix(y_test, knn_test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=knn_model.classes_)
disp.plot()
plt.title("KNN Confusion Matrix")
plt.show()


# Hyperparameter tuning for KNN
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Print best parameters and best score
print(f"Best Parameters for KNN: {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_}")

# Retrain KNN with best parameters
best_knn_model = grid_search.best_estimator_
best_knn_preds = best_knn_model.predict(X_val)
print("KNN Performance with Best Parameters on Validation Set:")
print(classification_report(y_val, best_knn_preds))

"""

Model: KNN classifier with optimized hyperparameters.

Evaluation:
Initial and optimized performance metrics on validation and test sets.
Confusion matrix for detailed performance visualization.

Hyperparameter Tuning:
Explored n_neighbors, weights, and metric for optimal performance.

This approach ensures a comprehensive evaluation and fine-tuning of the KNN model for the dataset

"""

# Decision Tree Classifier

The Decision Tree Classifier is applied to predict the target variable Cover_Type. 

This step includes:
Training and evaluating the Decision Tree model on validation and test sets.
Generating a confusion matrix for the test set.
Performing hyperparameter tuning to optimize the Decision Tree performance.

In [ ]:
# Initialize Decision Tree classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the Decision Tree model
dt_model.fit(X_train, y_train)

# Predictions on validation set
dt_preds = dt_model.predict(X_val)

# Print classification report for Decision Tree
print("Decision Tree Performance on Validation Set:")
print(classification_report(y_val, dt_preds))

# Evaluate on test set
dt_test_preds = dt_model.predict(X_test)
print("Decision Tree Test Performance:")
print(classification_report(y_test, dt_test_preds))

# Confusion Matrix for Decision Tree
cm = confusion_matrix(y_test, dt_test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dt_model.classes_)
disp.plot()
plt.title("Decision Tree Confusion Matrix")
plt.show()

# Hyperparameter tuning for Decision Tree
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid_search_dt.fit(X_train, y_train)

# Print best parameters and best score
print(f"Best Parameters for Decision Tree: {grid_search_dt.best_params_}")
print(f"Best Score: {grid_search_dt.best_score_}")

# Retrain Decision Tree with best parameters
best_dt_model = grid_search_dt.best_estimator_
best_dt_preds = best_dt_model.predict(X_val)
print("Decision Tree Performance with Best Parameters on Validation Set:")
print(classification_report(y_val, best_dt_preds))

"""
Model: Decision Tree Classifier with optimized hyperparameters

Evaluation:
Initial and optimized performance metrics on validation and test sets.
Confusion matrix for detailed performance visualization.

Hyperparameter Tuning:
Explored max_depth, min_samples_split, and min_samples_leaf for optimal performance.
This approach ensures a thorough evaluation and optimization of the Decision Tree model for the dataset.

"""

# Step 5: Model Evaluation and Visualizations

This step evaluates the classification models using various metrics and provides visualizations to understand their performance. The evaluation includes confusion matrices, ROC curves, and precision-recall (PR) curves.


In [ ]:
# confusion matrix
# Create confusion matrix for each model
models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "KNN": knn_model,
    "Decision Tree": dt_model
}

# Plot confusion matrices for each model
for name, model in models.items():
    test_preds = model.predict(X_test)
    cm = confusion_matrix(y_test, test_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.show()

#ROC Curve
# Binarize the labels for multi-class classification
y_test_bin = label_binarize(y_test, classes=range(len(y.unique())))

# Plot ROC curve for each model
for name, model in models.items():
    # Get the predicted probabilities for each class
    y_pred_prob = model.predict_proba(X_test)
    
    # Calculate ROC curve and AUC for each class
    fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_pred_prob.ravel())
    roc_auc = auc(fpr, tpr)
    
    # Plot the ROC curve
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for {name}')
    plt.legend(loc="lower right")
    plt.show()

# precision recall curve
# Plot Precision-Recall curve for each model
for name, model in models.items():
    # Get the predicted probabilities for each class
    y_pred_prob = model.predict_proba(X_test)
    
    # Calculate Precision-Recall curve and average precision score for each class
    precision, recall, _ = precision_recall_curve(y_test_bin.ravel(), y_pred_prob.ravel())
    avg_precision = average_precision_score(y_test_bin.ravel(), y_pred_prob.ravel())
    
    # Plot the Precision-Recall curve
    plt.figure()
    plt.plot(recall, precision, color='b', lw=2, label=f'PR curve (AP = {avg_precision:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve for {name}')
    plt.legend(loc="lower left")
    plt.show()

"""
Reports:

Logistic Regression Confusion Matrix: Class 4 has the highest correct predictions with 348 samples correctly classified.
Class 7 also performs well with 305 correct predictions. However, certain classes (e.g., Class 3 and Class 6) have fewer
correct predictions compared to others.

Random Forest Confusion Matrix: The model shows significant improvements in correctly classifying Classes 4, 5, and 7.
Random Forest achieves a higher accuracy compared to Logistic Regression, as reflected in the more concentrated diagonal values.

KNN Confusion matrix: The KNN model achieves moderate accuracy, but it struggles with class overlap and minority class predictions.
The model has a tendency to misclassify samples into neighboring classes, which is a known limitation of KNN due to its reliance on local neighbors.
These classes show better performance, with fewer misclassifications compared to others.

Decision tree confusion matrix: Class 4 has the highest correct prediction. and better prediction than knn.


The AUC is 0.44, 0.43, 0.45, 0.46, which is below 0.5. This indicates that the model performs worse than random guessing for this classification task.
The orange line lies close to the diagonal, representing a lack of separation between positive and negative classes.
The model struggles to distinguish between classes, possibly due to poor feature selection, imbalanced data, or inappropriate hyperparameter settings


Model Performance: All four models have difficulty identifying true positives relative to false positives, indicating potential issues 
in data representation, feature relevance, or class distribution.


"""


In [ ]:
# Initialize a dictionary to store the metrics for each model
model_metrics = {}

# Calculate and store metrics for each model
for name, model in models.items():
    # Predict on the test set
    test_preds = model.predict(X_test)
    
    # Calculate Accuracy
    accuracy = accuracy_score(y_test, test_preds)
    
    # Calculate classification report metrics
    report = classification_report(y_test, test_preds, output_dict=True)
    precision = report['macro avg']['precision']
    recall = report['macro avg']['recall']
    f1 = report['macro avg']['f1-score']
    
    # Get the predicted probabilities for ROC and PR curves
    y_pred_prob = model.predict_proba(X_test)
    
    # Calculate ROC AUC and PR AUC
    fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_pred_prob.ravel())
    roc_auc = auc(fpr, tpr)
    
    precision, recall, _ = precision_recall_curve(y_test_bin.ravel(), y_pred_prob.ravel())
    avg_precision = average_precision_score(y_test_bin.ravel(), y_pred_prob.ravel())
    
    # Store metrics
    model_metrics[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC AUC': roc_auc,
        'Average Precision': avg_precision
    }

# Convert metrics to a DataFrame for better visualization
metrics_df = pd.DataFrame(model_metrics).T
print(metrics_df)


"""

Logistic Regression shows poor precision and ROC AUC despite high recall. 
This suggests that the model predicts most samples as the dominant class, likely due to dataset imbalance.

Random Forest achieves the highest accuracy and F1-score among the models. 
However, its ROC AUC and average precision scores indicate challenges in separating some classes effectively.

KNN shows the lowest accuracy and F1-score, struggling with precision and recall for most classes.
Its average precision is slightly better than Logistic Regression and Random Forest, but overall performance is suboptimal.

The Decision Tree model performs better than Logistic Regression and KNN but falls short of Random Forest in terms of 
accuracy and F1-score. Its precision and recall indicate potential overfitting to specific classes.


Random Forest outperforms other models in terms of accuracy (86.03%) and F1-score (85.58%). 
However, it struggles with certain classes, as indicated by its low ROC AUC (42.72%).

Logistic Regression performs poorly, likely due to its linear nature and sensitivity to class imbalance.

KNN shows the weakest performance, with the lowest accuracy and F1-score, indicating it may not be suitable for this dataset.

Decision Tree provides decent performance but lacks the robustness of Random Forest.

Random Forest is the best-performing model and should be further optimized using hyperparameter tuning and feature engineering.

"""
